In [3]:
import cv2
import os
import urllib.request
import time

In [8]:
face_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
smile_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_smile.xml"

face_file = "haarcascade_frontalface_default.xml"
smile_file = "haarcascade_smile.xml"

if not os.path.exists(face_file):
    urllib.request.urlretrieve(face_url, face_file)

if not os.path.exists(smile_file):
    urllib.request.urlretrieve(smile_url, smile_file)

save_path = 'smiles'
os.makedirs(save_path, exist_ok=True)

last_capture_time = 0
capture_delay = 2
smile_count = 0

def detect(img, faceCascade, smileCascade):
    color = {"blue":(255,0,0), "green":(0,255,0)}

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = faceCascade.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=5,
        minSize=(60, 60)
    )

    smile_detected = False

    for (x, y, w, h) in faces:
        cv2.rectangle(img, (x,y), (x+w,y+h), color['blue'], 2)

        roi_gray = gray[y + h//2:y + h, x:x+w]

        smiles = smileCascade.detectMultiScale(
            roi_gray,
            scaleFactor=1.3,
            minNeighbors=30,
            minSize=(25, 25)
        )

        if len(smiles) > 0:
            smile_detected = True
            cv2.putText(img, "Smile", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color['green'], 2)

    return img, smile_detected


faceCascade = cv2.CascadeClassifier(face_file)
smileCascade = cv2.CascadeClassifier(smile_file)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img, smile_detected = detect(frame, faceCascade, smileCascade)

    if smile_detected:
        current_time = time.time()

        if current_time - last_capture_time > capture_delay:
            smile_count += 1

            file_name = f"{save_path}/smile_{smile_count}.jpg"
            cv2.imwrite(file_name, frame)

            print(f"Saved: {file_name}")

            last_capture_time = current_time

    cv2.putText(img, f"Smile Count: {smile_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("Smile Detection", img)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()

Saved: smiles/smile_1.jpg
Saved: smiles/smile_2.jpg
Saved: smiles/smile_3.jpg
Saved: smiles/smile_4.jpg
Saved: smiles/smile_5.jpg
